# 02. Modular Arithmetic and Fast Power

"Return the answer modulo 10⁹ + 7" appears in many counting problems. This notebook shows why, and how to compute powers and inverses efficiently.

**Learning order:** WHY MOD → RULES → FAST POWER → pow(a, b, m) → MODULAR INVERSE → nCr MOD p → MATRIX POWER → INTERVIEW THINKING

## 1. Why modulo?

Counts grow huge (permutations of 1000 items). Other languages overflow 64 bits. So problems ask for the answer **mod `M`** (usually `10⁹ + 7`, a prime). Python never overflows but big numbers are slow, so we still reduce at every step.

## 2. Rules

- `(a + b) % M = ((a % M) + (b % M)) % M`
- `(a * b) % M = ((a % M) * (b % M)) % M`
- `(a - b) % M`: in Python the result is already non-negative. In C++/Java add `M` before the final `%`.
- **Division is different:** `(a / b) % M` needs the **modular inverse** of `b`.

In [1]:
MOD = 10**9 + 7
print((10**18 + 5) % MOD, (-7) % 5, (123456789 * 987654321) % MOD)   # 54 3 259106859

54 3 259106859


## 3. Fast exponentiation: `a^b` in O(log b)

`a^b = (a^(b/2))²` if `b` is even, `a · a^(b-1)` if odd. Or iteratively using the **binary digits of `b`**: square `a` each step, multiply the result when the bit is 1.

In [2]:
def power_mod(a, b, m):
    result = 1
    a %= m
    while b:
        if b & 1:
            result = result * a % m
        a = a * a % m
        b >>= 1
    return result

print(power_mod(2, 10, 1000), power_mod(3, 200, MOD), pow(3, 200, MOD))   # 24 136318165 136318165

24 136318165 136318165


## 4. Modular inverse (Fermat's little theorem)

For a **prime** `M` and `a` not divisible by `M`: `a^(M-1) ≡ 1 (mod M)`, so `a^(-1) ≡ a^(M-2) (mod M)`. Python 3.8+ also has `pow(a, -1, M)`.

Use it to divide: `(a / b) % M = a * inverse(b) % M`.

In [3]:
def mod_inverse(a, m=MOD):
    return pow(a, m - 2, m)

inv2 = mod_inverse(2)
print(inv2, 2 * inv2 % MOD, pow(2, -1, MOD))   # 500000004 1 500000004

500000004 1 500000004


## 5. nCr mod p with precomputed factorials

`C(n, r) = n! / (r! (n - r)!)`. Precompute factorials and inverse factorials once, then each query is O(1).

In [4]:
N = 1000
fact = [1] * (N + 1)
for i in range(1, N + 1):
    fact[i] = fact[i - 1] * i % MOD
inv_fact = [1] * (N + 1)
inv_fact[N] = pow(fact[N], MOD - 2, MOD)
for i in range(N, 0, -1):
    inv_fact[i - 1] = inv_fact[i] * i % MOD

def ncr(n, r):
    if r < 0 or r > n:
        return 0
    return fact[n] * inv_fact[r] % MOD * inv_fact[n - r] % MOD

print(ncr(5, 2), ncr(10, 5), ncr(1000, 500))   # 10 252 159835829

10 252 159835829


## 6. Matrix exponentiation: Fibonacci in O(log n)

`[F(n+1), F(n)] = [[1,1],[1,0]]^n · [1, 0]`. Raise the 2×2 matrix to the power `n` with fast exponentiation. Useful for **linear recurrences** with huge `n` (10¹⁸).

In [5]:
def mat_mult(A, B, m=MOD):
    return [[(A[0][0]*B[0][0] + A[0][1]*B[1][0]) % m, (A[0][0]*B[0][1] + A[0][1]*B[1][1]) % m],
            [(A[1][0]*B[0][0] + A[1][1]*B[1][0]) % m, (A[1][0]*B[0][1] + A[1][1]*B[1][1]) % m]]

def fib_fast(n):
    result, base = [[1, 0], [0, 1]], [[1, 1], [1, 0]]
    while n:
        if n & 1:
            result = mat_mult(result, base)
        base = mat_mult(base, base)
        n >>= 1
    return result[0][1]

print(fib_fast(10), fib_fast(90) == 2880067194370816120 % MOD, fib_fast(10**18))   # 55 True 209783453

55 True 209783453


## 7. Interview thinking

**Signals:** "return the answer modulo 10⁹+7", "power with huge exponent", "number of ways" for large `n`, "nCr".

**Say:** "I'll apply the modulo after every multiplication/addition to keep numbers small. For division I use the modular inverse via Fermat. Power in O(log n)."

**Common mistakes:** applying `%` only at the end, subtracting without normalizing (C++/Java), using `/` instead of the inverse, using Fermat when the modulus is **not prime**, `pow(a, b, m)` with negative `b` on old Python versions.

## Quick Revision

- `(a*b) % M` can be reduced at any step. Division → modular inverse `a^(M-2)`.
- Fast power: square the base, multiply on set bits. Python: `pow(a, b, m)`.
- nCr: precompute `fact` and `inv_fact`.
- Matrix power for linear recurrences.